# Lab 1 — Wire It Up and Test It

Welcome! Today you're building a **game controller** out of parts. By the end of this notebook you'll have proof that both of your inputs are wired correctly and working.

Here's the journey:

1. **Wire** your two inputs to the computer (the little board in front of you).
2. **Tell Python** which pins your wires are plugged into.
3. **Read** the raw numbers coming from your inputs.
4. **Tame** those numbers into a range games can understand.
5. **Live test** everything at once — with a helper watching for wires in the wrong holes.

### How to use this notebook

A notebook is made of **cells**. Some are text (like this one), some are code. Click a code cell and press **Shift + Enter** to run it. The output appears right below the cell. Run cells **top to bottom** — later cells depend on earlier ones.

You will only run the cells that match **your two inputs** — it's fine to skip the sections for inputs you didn't pick.

> This notebook talks directly to the pins, so Jupyter has to be started with admin rights on the Pi (your facilitator has done this, but for reference: `sudo jupyter notebook --allow-root`).

## What are GPIO and i2c?

Before wiring anything, meet the two ways your inputs will talk to the computer.

### GPIO — one wire, one simple signal

**GPIO** stands for **G**eneral **P**urpose **I**nput/**O**utput. A GPIO pin is a wire connected straight into the computer's brain that can be given a simple job. Today its job is *input*: the computer constantly feels whether there is voltage on the pin, and reports one of exactly two answers:

- about **3.3 volts** → reads as **1**
- about **0 volts** → reads as **0**

That's the whole trick. A button is just a switch that changes the voltage on its pin when you press it — so "is the button pressed?" becomes "did the number on pin 8 change?". Your buttons, the encoder's three signals, and the joystick's press-down switch all work this way: one wire each, on or off.

### i2c — two wires, a whole conversation

Some inputs have more to say than on/off. A joystick's position is a number in the thousands, and the tilt sensor measures gravity in three directions at once. You can't say "13,204" with a single on/off wire.

**i2c** is a miniature language for exactly this. It uses **two** wires:

- **SDA** (data) — the wire the actual numbers travel over, one bit at a time
- **SCL** (clock) — a steady tick-tick-tick that keeps both sides reading the bits at the same rhythm

The clever part: i2c is built for **sharing**. Every i2c chip has an *address*, like houses on a street — the ADS1115 (the joystick's translator chip) answers at address `0x48`, the tilt sensor at `0x68`. The computer says "address 0x68, tell me your readings", and only that chip replies. That's why *both* chips plug into the *same* two pins and never get confused.

### Why must i2c use pins 3 and 5?

GPIO pins are interchangeable — any of them can feel a voltage, which is why you get to choose your button pins. i2c is different: speaking it needs dedicated hardware inside the computer's chip, and that hardware is physically wired out to **pin 3 (SDA)** and **pin 5 (SCL)** — those exact pins, chosen by the board designers. Run `gpio readall` in a terminal and you'll see those two pins are even *named* `SDA.0` and `SCL.0`. So the ADS1115 and the tilt sensor must both connect there — there is nowhere else the computer can hear them.

### The pin map

Here is the whole 40-pin header. Power and ground pins feed your modules, pins 3/5 are the shared i2c pair, and every pin marked GPIO is yours to choose from. (Pins 26, 27 and 28 are wired to other jobs on this board — leave them alone.)

```
        3.3V power    1 |  2    5V power
      i2c data SDA    3 |  4    5V power
     i2c clock SCL    5 |  6    ground
              GPIO    7 |  8    GPIO
            ground    9 | 10    GPIO
              GPIO   11 | 12    GPIO
              GPIO   13 | 14    ground
              GPIO   15 | 16    GPIO
        3.3V power   17 | 18    GPIO
              GPIO   19 | 20    ground
              GPIO   21 | 22    GPIO
              GPIO   23 | 24    GPIO
            ground   25 | 26    (reserved)
        (reserved)   27 | 28    (reserved)
              GPIO   29 | 30    ground
              GPIO   31 | 32    GPIO
              GPIO   33 | 34    ground
              GPIO   35 | 36    GPIO
              GPIO   37 | 38    GPIO
            ground   39 | 40    GPIO
```

So the GPIO pins you may pick from are: **7, 8, 10, 11, 12, 13, 15, 16, 18, 19, 21, 22, 23, 24, 29, 31, 32, 33, 35, 36, 37, 38, 40** — and the helper will tell you if you pick one that isn't on the list.

## Wiring reference

The double row of metal pins on the board is called the **header**. Every pin has a number: pin 1 is marked on the board, odd numbers run down one side (1, 3, 5, ...) and even numbers down the other (2, 4, 6, ...). We always use these **physical pin numbers** — count them, don't guess!

**The golden rule: every module gets power from a 3.3V pin, never 5V.** 5V can damage the computer's inputs.

Every input module needs three kinds of wires:

- **VCC / +** → a 3.3V pin (power)
- **GND** → a GND pin (the return path)
- **signal wires** → the pins listed below (these carry the actual information)

| Your input | Signal wires | Where they go |
|---|---|---|
| **Joystick** (HW-504) | SW | any free GPIO pin (we suggest **36**) |
| | VRx, VRy | to the **ADS1115** board: LEFT side uses A3 and A2, RIGHT side uses A1 and A0 |
| **Rotary encoder** (KY-040) | CLK, DT, SW | any three free GPIO pins (we suggest **23, 21, 19**) |
| **Two buttons** (KY-004) | OUT (one per button) | any two free GPIO pins (we suggest **8 and 10**) |
| **Tilt sensor** (MPU-6050) | SDA, SCL | **pin 3** and **pin 5** (these are special data pins) |

Two of the parts don't plug straight into numbered pins:

- The **joystick's stick** speaks "analog" (a smoothly changing voltage). The computer only understands on/off, so a translator chip called the **ADS1115** reads the voltage and reports the number over the data pins (3 and 5). The joystick's *press-down button* is a normal on/off signal, so that one **does** get its own numbered pin.
- The **tilt sensor** has its own tiny brain and also talks over the data pins (3 and 5).

Wire everything up now. When you're done, continue below.

## Step 1 — Tell Python where your wires go

Your controller has a **LEFT** input and a **RIGHT** input (from the player's point of view, holding the controller).

Edit the cell below so it matches **your** build:

1. Set `LEFT_INPUT` and `RIGHT_INPUT` to the names of your two inputs.
2. Set the pin numbers to the pins **you actually plugged the signal wires into**.

The commented examples at the bottom of the cell show the exact shape for each input type — copy the ones you need up into place. When it looks right, run the cell (Shift + Enter). Nothing visible happens yet; you're just storing your answers.

In [2]:
# ================ YOUR CONTROLLER SETUP ================
# Edit these four lines to match YOUR wiring.
# (You'll copy this whole cell into Lab 2 later - it IS your controller.)

LEFT_INPUT = "joystick"
LEFT_PINS = {"SW": 36}

RIGHT_INPUT = "buttons"
RIGHT_PINS = {"A": 8, "B": 10}

# ------------- copy from these examples -------------
# joystick:   "joystick"   {"SW": 36}
# encoder:    "encoder"    {"CLK": 23, "DT": 21, "SW": 19}
# buttons:    "buttons"    {"A": 8, "B": 10}
# tilt:       "tilt"       {}
#
# The numbers are PHYSICAL pin numbers on the header.
# Count them on the board - don't trust the defaults!

Now hand your answers to the helper. It will set up every pin, look for the translator chips it needs, and complain **in plain English** if something seems wrong — read its messages!

In [3]:
import workshopHelpers as ws

ws.connect(LEFT_INPUT, LEFT_PINS, RIGHT_INPUT, RIGHT_PINS)

setting up your controller...
  LEFT  = joystick (SW -> pin 36)
  RIGHT = buttons  (A -> pin 8, B -> pin 10)
  found the ADS1115 (the chip that reads the joystick) - good!
all set! I'm also quietly watching the other 21 header pins in case a wire landed in the wrong hole.


## Step 2 — First readings

Time to hear what your inputs are actually saying. Below there's one small section per input type. **Only run the sections for your two inputs.**

The trick for each one: run the cell, *do something to the input* (push, press, turn, tilt), then run the cell **again** and watch the number change. If the number never changes, a wire is loose or on the wrong pin.

### Joystick

The stick reports two numbers, **X** (left-right) and **Y** (up-down), each somewhere between **0 and about 26400**. Centered, both sit near the middle (~13000).

The cells below say `"left"` — change it to `"right"` if your joystick is your RIGHT input.

In [7]:
x, y = ws.read_joystick("left")     # change to "right" if your joystick is on the right
print("X:", x, "   Y:", y)
print("now HOLD the stick to one side and run this cell again!")

X: 226    Y: 12638
now HOLD the stick to one side and run this cell again!


### Buttons (also for the joystick's press and the encoder's press)

A button is the simplest input there is: it answers one question — *"am I being held down right now?"* — with `True` or `False`.

(When you ran the connect cell, the helper quietly noted what each button looks like while *nobody* is touching it, so "pressed" means "different from that".)

Put **your** pin numbers in the cell below. This works for any press-type pin: your A/B buttons, your joystick's SW pin, or your encoder's SW pin.

In [9]:
print("pin 8 pressed:", ws.read_button(8))     # <-- use YOUR pin numbers
print("pin 10 pressed:", ws.read_button(10))
print("now HOLD a button down and run this cell again!")

pin 8 pressed: True
pin 10 pressed: True
now HOLD a button down and run this cell again!


### Rotary encoder

The knob doesn't know where it's pointing — it only reports **clicks**: each notch you turn counts **+1** one way and **-1** the other way. The helper adds them up in the background, so you just ask for the running total (the "dial position").

Change `"left"` to `"right"` if your encoder is your RIGHT input.

In [10]:
print("dial position:", ws.read_dial("left"))   # change to "right" if needed
print("now TURN the knob a few clicks and run this cell again!")

RuntimeError: 

Your LEFT input is set to 'joystick', not 'encoder'.


### Tilt sensor

The tilt sensor feels **gravity**. It reports how much of gravity's pull it feels in each direction, measured in **g** (1 g = full Earth gravity). Tip the sensor and gravity "leans" into a different direction — that's how it knows it's tilted.

Don't worry about the exact numbers while it's flat; we'll zero it out in the next step.

In [ ]:
forward, right = ws.read_tilt()
print("forward tilt:", round(forward, 2), "g    right tilt:", round(right, 2), "g")
print("now HOLD it tilted to one side and run this cell again!")

## Step 3 — Put bounds on the numbers

Raw numbers like `26314` or `0.43 g` mean nothing to a game. Games expect every stick-like input in the same tidy range:

- **-1.0** = all the way one direction
- **0.0** = resting in the middle
- **+1.0** = all the way the other direction

Two tools get us there:

- **Rescaling** — measure the lowest and highest number your input actually produces, then squeeze that range into -1 to +1.
- **A deadzone** — no part is perfect: a "centered" stick actually reads 0.02, then 0.04, then 0.01... A deadzone snaps anything tiny to exactly 0 so your character doesn't drift when you're not touching anything.

**Buttons need none of this** — `True`/`False` is already perfect. Run only the cells for your inputs.

### Joystick bounds

First, find *your* joystick's real range. Run the next cell and immediately roll the stick around its full circle until it finishes.

In [11]:
import time

print("roll the joystick around its FULL circle for 5 seconds... GO!")
xs = []
ys = []
for i in range(100):
    x, y = ws.read_joystick("left")     # change to "right" if needed
    xs.append(x)
    ys.append(y)
    time.sleep(0.05)

JOY_LOW = min(xs + ys)
JOY_HIGH = max(xs + ys)
print("done! your joystick reads between", JOY_LOW, "and", JOY_HIGH)

roll the joystick around its FULL circle for 5 seconds... GO!
done! your joystick reads between 10 and 26389


Now apply both tools: rescale the raw reading using *your* measured bounds, then deadzone it. Hold the stick somewhere while you run this — and try it resting, too. Resting should print exactly `0.0` for both.

The last line stores your calibration so the live test (and Lab 2) use it automatically.

In [12]:
x, y = ws.read_joystick("left")          # change to "right" if needed

x = ws.rescale(x, JOY_LOW, JOY_HIGH)     # squeeze into -1 .. +1
y = ws.rescale(y, JOY_LOW, JOY_HIGH)
x = ws.deadzone(x, 0.08)                 # tiny wobbles become exactly 0
y = ws.deadzone(y, 0.08)

print("X:", round(x, 2), "   Y:", round(y, 2))

ws.remember_bounds("left", JOY_LOW, JOY_HIGH)   # change to "right" if needed

X: 0.0    Y: 0.0
remembered! your left joystick reads 10 (one end) to 26389 (other end).


### Encoder bounds

The dial position can grow forever (+37, +38, ...), but a game control can't. So we put a **bound** on it: **8 clicks** in either direction counts as "all the way". 4 clicks up = `+0.5`, 8 or more clicks down = `-1.0`. Turn the knob between runs and watch both numbers.

In [ ]:
clicks = ws.read_dial("left")            # change to "right" if needed
bounded = ws.read_dial_percent("left")

print("dial position:", clicks, "clicks  ->  as a game control:", round(bounded, 2))

### Tilt zeroing

Even sitting "flat", the tilt sensor feels gravity — so flat doesn't read zero. The fix is to **remember what flat looks like** and measure every tilt *relative to that*. (The connect cell already did this once; here you can redo it whenever you like — useful if a player holds the controller at their own comfortable angle. *Their* comfortable position becomes the new zero. That's accessibility in action.)

Hold the controller in its resting position, run the cell, then tilt and run it again.

In [ ]:
ws.remember_flat()    # ONLY run this line while holding the controller still, in its resting position
print("zeroed! this position now counts as 'not tilting'.")

x, y = ws.read_tilt_percent()
print("tilt as a stick position ->  X:", round(x, 2), "  Y:", round(y, 2))

## Step 4 — The live test

Time for the final exam. The next cell opens a **live dashboard** showing both of your inputs updating in real time. Work through your controller: push, press, turn, tilt — everything.

- Each part shows an arrow like `<- press it!` until you've made it work, then it flips to `OK`.
- When a whole input checks out, its header says **ALL WORKING!**
- The helper is also watching **every pin you did NOT list**. If a press shows up on, say, pin 38 when you told us pin 36, a big `!!` warning appears — that means a wire is one hole off from where you think it is. Fix the wire (or fix the number in your Step 1 cell, then re-run Step 1 and the connect cell).

Press the **square stop button** next to the cell (or the toolbar) when both inputs say ALL WORKING.

In [13]:
ws.live_feed()

live test stopped.
note: I saw activity on pins that are not in your setup: pin 21, pin 23
if an input never showed up above, its wire is probably on one of those pins.


## Done — your inputs are verified!

You just did real engineering: wired hardware, read raw sensor data, calibrated it, and verified the whole system end to end.

**Before you leave this notebook:** select your **Step 1 setup cell** (the one starting with `YOUR CONTROLLER SETUP`) and copy it — you'll paste it at the top of **Lab 2**, where those same verified pins become a real gamepad that plays SuperTuxKart.